In [ ]:
# [Cell 1/4] Hardware Accelerator Detection (Colab G4 96GB / A100 / L4 / T4 / Local RTX) & Batch Auto-Scaling
# !pip install -q ultralytics onnx onnxruntime openvino

import json
import os

NINJANVR_SECURITY_CLASSES = [
    "person", "face", "car", "motorcycle", "bicycle",
    "truck", "bus", "license_plate", "dog", "cat",
    "bird", "package", "waste_bin", "backpack", "umbrella",
]

def detect_gpu_and_batch_size():
    gpu_name = "CPU"
    vram_gb = 0.0
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            gpu_name = props.name
            vram_gb = round(props.total_memory / (1024 ** 3), 1)
    except Exception:
        pass

    # Auto-scale batch size by VRAM: G4 96GB -> 128, A100 40GB -> 64, RTX 4070 Ti Super / L4 16-24GB -> 32, T4 -> 16, CPU -> 8
    if vram_gb >= 80.0 or "G4" in gpu_name or "6000" in gpu_name:
        batch_size = 128
    elif vram_gb >= 40.0 or "A100" in gpu_name:
        batch_size = 64
    elif vram_gb >= 16.0 or "L4" in gpu_name or "4070" in gpu_name:
        batch_size = 32
    elif vram_gb >= 8.0 or "T4" in gpu_name:
        batch_size = 16
    else:
        batch_size = 8
    return gpu_name, vram_gb, batch_size

GPU_NAME, VRAM_GB, BATCH_SIZE = detect_gpu_and_batch_size()
print(json.dumps({"gpu": GPU_NAME, "vram_gb": VRAM_GB, "batch_size": BATCH_SIZE, "nc": len(NINJANVR_SECURITY_CLASSES)}))


In [ ]:
# [Cell 2/4] Dataset Ingestion (ninjanvr-dataset.zip) & Synthetic CCTV Night-IR Augmentation (35% frames)
import math
import random
import zipfile
from pathlib import Path

MODE = os.environ.get("NINJANVR_TRAIN_MODE", "finetune")  # 'finetune' or 'base'
DATASET_ZIP = os.environ.get("NINJANVR_DATASET_ZIP", "ninjanvr-dataset.zip")
DATASET_DIR = Path("./dataset_workspace")

def rgb_to_night_ir_luma(r: int, g: int, b: int, r2_norm: float, vignette_strength: float = 0.35) -> int:
    # ITU-R BT.601 luma Y = 0.299R + 0.587G + 0.114B + radial IR LED vignette falloff
    y = 0.299 * r + 0.587 * g + 0.114 * b
    vignette = max(0.15, min(1.0, 1.0 - vignette_strength * r2_norm))
    return max(0, min(255, int(round(y * vignette))))

if Path(DATASET_ZIP).exists():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(DATASET_DIR)
    print(f"Extracted {DATASET_ZIP} into {DATASET_DIR}")
else:
    print(f"Ready for dataset archive {DATASET_ZIP} (mode={MODE}, night_ir_prob=0.35)")


In [ ]:
# [Cell 3/4] Dual-Mode YOLO Trainer (base vs finetune with freeze=10 backbone layers)
MODEL_ARCH = os.environ.get("NINJANVR_MODEL_ARCH", "yolo11s.pt")
IMGSZ = int(os.environ.get("NINJANVR_IMGSZ", "640"))
FREEZE_LAYERS = 10 if MODE == "finetune" else 0
EPOCHS = int(os.environ.get("NINJANVR_EPOCHS", "30" if MODE == "finetune" else "100"))

TRAIN_CONFIG = {
    "mode": MODE,
    "model": MODEL_ARCH,
    "data": str(DATASET_DIR / "data.yaml"),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH_SIZE,
    "freeze": FREEZE_LAYERS,
    "perspective": 0.0005,
    "mosaic": 0.5 if MODE == "finetune" else 1.0,
}
print("Training configuration:", json.dumps(TRAIN_CONFIG, indent=2))

# Uncomment or run in Colab when dataset_workspace/data.yaml is mounted:
# from ultralytics import YOLO
# model = YOLO(MODEL_ARCH)
# model.train(**{k: v for k, v in TRAIN_CONFIG.items() if k != 'mode'})


In [ ]:
# [Cell 4/4] Export Static ONNX (opset=17) & OpenVINO FP16 IR (.xml/.bin) + Optional Hot-Reload Upload
import base64
import urllib.request

MODEL_NAME = os.environ.get("NINJANVR_MODEL_NAME", "ninjanvr_security_15cl")
NVR_URL = os.environ.get("NINJANVR_URL", "")
NVR_TOKEN = os.environ.get("NINJANVR_TOKEN", "")

EXPORT_SPEC = {
    "onnx": f"{MODEL_NAME}.onnx",
    "openvino_xml": f"{MODEL_NAME}.xml",
    "openvino_bin": f"{MODEL_NAME}.bin",
    "labelmap": "ninjanvr_security_15cl.txt",
    "opset": 17,
    "input_shape": [1, 3, IMGSZ, IMGSZ],
    "output_shape": [1, 19, 8400 if IMGSZ == 640 else 2100],
    "half": True,
}
Path("ninjanvr_security_15cl.txt").write_text("\n".join(NINJANVR_SECURITY_CLASSES) + "\n", encoding="utf-8")
print("Export specification:", json.dumps(EXPORT_SPEC, indent=2))
